# CSE457 – Lab 4: Tổng hợp tiếng nói (TTS)

- **Họ tên:** Phạm Duy Long  **MSSV:** 2251061829  **Lớp:** 64CNTT1
- **Bài toán:** mini TTS miền hẹp "thông báo xếp hàng" – câu mẫu *"Mời khách hàng số 23 đến quầy 5."*
- **Ba hướng tổng hợp được so sánh:** (1) ghép nối word-unit + unit selection + cross-fade, (2) tổng hợp tham số dựa trên HMM (sinh quỹ đạo bằng MLPG + vocoder source–filter), (3) TTS học sâu pretrained chạy offline (Piper/VITS).
- **Cách chạy:** đặt inventory vào `lab4_data/units/` (`python record_units.py`), rồi *Kernel → Restart & Run All*. Backend HMM tự huấn luyện từ inventory; neural TTS cần model Piper đặt sẵn trong `lab4_data/neural/` (xem README). Nếu chưa có model neural, notebook vẫn chạy và đánh dấu phần đó là "chưa chạy".

Mục lục: A (normalization + inventory) → B (ghép thẳng) → C (unit selection + cross-fade) → D (prosody, source–filter) → E (HMM) → F (neural) → G (đánh giá) → câu hỏi báo cáo.

In [ ]:
import sys, time, math, warnings, re, unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import soundfile as sf
from scipy.fft import rfft

warnings.filterwarnings("ignore")
sys.path.insert(0, ".")
import hmm_tts_backend as HMMB          # backend HMM (đi kèm repo)

FS = 16000
DATA = Path("lab4_data"); UNITS = DATA / "units"
OUT = Path("audio_output"); FIG = Path("figures"); RES = Path("results")
for p in (OUT, FIG, RES, OUT / "xfade", OUT / "prosody", OUT / "blind"):
    p.mkdir(parents=True, exist_ok=True)

# ---- toàn bộ tham số thí nghiệm ở MỘT chỗ (để tái lập) ----
CFG = dict(
    speaker="long",
    target_rms=0.08,        # RMS chuẩn hoá cho mỗi unit
    trim_db=35,             # ngưỡng cắt silence (librosa.effects.trim)
    edge_margin_ms=10,      # chừa lại 10 ms mỗi phía sau khi trim
    zc_snap_ms=5,           # trượt biên cắt tới điểm zero-crossing gần nhất
    join_ms=25,             # độ dài đoạn lấy phổ ở biên để tính join cost
    xfade_ms=15,            # cross-fade mặc định
    xfade_grid=[0, 5, 15, 30],
    lam=1.0,                # λ cân bằng join cost
    w_spec=1.0, w_rms=0.5, w_f0=0.5,   # trọng số các thành phần join cost
    w_dur=1.0, w_f0_target=0.5,        # trọng số target cost
    dur_tol=0.15, f0_tol_st=2.0,       # dung sai chuẩn hoá (ln tỷ lệ duration, semitone)
    last_word_dur=1.15, last_word_f0=0.90,   # kéo dài + hạ giọng từ cuối câu (phrase-final)
)
SENTENCES = [
    "Mời khách hàng số 23 đến quầy 5.",     # câu mẫu của đề
    "Mời khách hàng số 108 đến quầy 2.",
    "Mời khách hàng số 47 đến quầy 9.",
]
np.random.seed(0)
print("librosa", librosa.__version__, "| numpy", np.__version__)

## A. Text normalization và inventory

Miền hẹp: mọi chuỗi số được đọc **từng chữ số** (ví dụ "23" → *hai ba*, "108" → *một không tám*). Quy tắc này là mã số/thứ tự, không phải số đếm; đây chính là chỗ cùng một chuỗi ký tự có thể cần ngữ cảnh để chọn cách đọc (mục 3.2 của đề).

In [ ]:
DIGIT = {"0": "không", "1": "một", "2": "hai", "3": "ba", "4": "bốn",
         "5": "năm", "6": "sáu", "7": "bảy", "8": "tám", "9": "chín"}

def normalize_queue_text(text):
    # lowercase, bỏ dấu câu, tách token, đọc từng chữ số
    text = text.lower().replace(".", "").replace(",", "")
    out = []
    for tok in text.split():
        if tok.isdigit():
            out.extend(DIGIT[c] for c in tok)
        else:
            out.append(tok)
    return out

# từ (có dấu) -> nhãn file (không dấu) của inventory
UNIT_FILE = {"mời": "moi", "khách": "khach", "hàng": "hang", "số": "so",
             "không": "khong", "một": "mot", "hai": "hai", "ba": "ba", "bốn": "bon",
             "năm": "nam", "sáu": "sau", "bảy": "bay", "tám": "tam", "chín": "chin",
             "đến": "den", "quầy": "quay"}

for s in SENTENCES:
    print(f"{s!r:40s} -> {normalize_queue_text(s)}")

need = sorted({w for s in SENTENCES for w in normalize_queue_text(s)})
missing_vocab = [w for w in need if w not in UNIT_FILE]
assert not missing_vocab, f"Token ngoài miền hẹp: {missing_vocab}"
print("\nTất cả token đều nằm trong inventory 16 từ.")

In [ ]:
if not UNITS.exists() or not list(UNITS.glob("*.wav")):
    raise FileNotFoundError("Chưa có inventory. Chạy `python record_units.py` (16 từ x 3 take) "
                            "hoặc chép WAV vào lab4_data/units/<nhãn>_01.wav ...")

def rms(x):
    return float(np.sqrt(np.mean(np.asarray(x, dtype=np.float64) ** 2) + 1e-12))

def match_rms(x, target=0.08):
    y = x * (target / rms(x))
    return np.clip(y, -0.99, 0.99).astype(np.float32)

def snap_zero_crossing(x, idx, tol):
    # trượt idx tới điểm đổi dấu gần nhất trong ±tol mẫu (giảm click khi cắt/ghép)
    lo, hi = max(idx - tol, 1), min(idx + tol, len(x) - 1)
    zc = np.where(np.signbit(x[lo - 1:hi - 1]) != np.signbit(x[lo:hi]))[0]
    if len(zc) == 0:
        return idx
    return int(lo + zc[np.argmin(np.abs(lo + zc - idx))])

def process_unit(path):
    raw, sr = sf.read(str(path), dtype="float32")
    if raw.ndim > 1:
        raw = raw.mean(axis=1)
    assert sr == FS, f"{path.name}: Fs = {sr}, cần {FS} Hz"
    _, (s, e) = librosa.effects.trim(raw, top_db=CFG["trim_db"], frame_length=400, hop_length=160)
    m = int(CFG["edge_margin_ms"] * FS / 1000)
    tol = int(CFG["zc_snap_ms"] * FS / 1000)
    s, e = max(s - m, 0), min(e + m, len(raw))
    s, e = snap_zero_crossing(raw, s, tol), snap_zero_crossing(raw, e, tol)
    x = raw[s:e]
    return match_rms(x, CFG["target_rms"]), rms(raw), len(raw) / FS

INV, rows = {}, []
for word, lab in UNIT_FILE.items():
    files = sorted(UNITS.glob(f"{lab}_*.wav"))
    if not files:
        print(f"  [thiếu] chưa có file cho '{word}' ({lab}_*.wav)")
    for k, f in enumerate(files, 1):
        x, rms_raw, dur_raw = process_unit(f)
        INV.setdefault(word, []).append(dict(word=word, label=lab, take=k, file=str(f), x=x))
        rows.append(dict(label=lab, word=word, speaker=CFG["speaker"], take=k,
                         duration_ms=round(1000 * len(x) / FS, 1), rms_raw=round(rms_raw, 4),
                         rms_norm=round(rms(x), 4), path=str(f)))
meta = pd.DataFrame(rows)
meta.to_csv(DATA / "metadata.csv", index=False)

missing = [w for w in need if w not in INV]
assert not missing, f"Thiếu unit cho: {missing}"
few = [w for w in INV if len(INV[w]) < 2]
print(f"{len(INV)} từ, {len(meta)} unit; từ có <2 take (unit selection sẽ không có lựa chọn): {few or 'không'}")
meta.head(8)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
ax[0].hist(meta.duration_ms, bins=15, color="#4c72b0"); ax[0].set_title("Phân bố duration unit (ms)"); ax[0].set_xlabel("ms")
ax[1].hist(meta.rms_raw, bins=15, color="#dd8452"); ax[1].set_title("RMS gốc (trước chuẩn hoá)"); ax[1].set_xlabel("RMS")
g = meta.groupby("word").duration_ms.mean().sort_values()
ax[2].barh(g.index, g.values, color="#55a868"); ax[2].set_title("Duration trung bình theo từ (ms)")
plt.tight_layout(); plt.savefig(FIG / "A_inventory_stats.png", dpi=130); plt.show()

# kiểm tra bất thường: duration/RMS lệch > 2.5 σ so với các take cùng từ hoặc toàn tập
z_d = (meta.duration_ms - meta.duration_ms.mean()) / (meta.duration_ms.std() + 1e-9)
z_r = (meta.rms_raw - meta.rms_raw.mean()) / (meta.rms_raw.std() + 1e-9)
print("Unit bất thường (|z|>2.5):")
print(meta[(z_d.abs() > 2.5) | (z_r.abs() > 2.5)][["word", "take", "duration_ms", "rms_raw"]].to_string(index=False) or "  không có")

**Nhận xét (điền sau khi xem hình):** duration các unit nằm trong khoảng …; RMS gốc chênh lệch … lần giữa unit to nhất và nhỏ nhất, nên cần match RMS trước khi ghép để tránh nhảy mức tại biên. Unit bất thường (nếu có): …

### Đặc trưng cho unit selection (F0, RMS, phổ ở biên)

Mỗi unit được gắn: duration, F0 trung vị (pyin), và ở hai đầu: log-phổ 25 ms (cửa sổ Hann), mức RMS (dB), F0 biên. Đây là các đại lượng dùng cho **target cost** và **join cost**.

In [ ]:
NFFT = 512
def edge_spec(seg):
    w = np.hanning(len(seg))
    A = rfft(seg * w, n=NFFT)
    return 20 * np.log10(np.maximum(np.abs(A), 1e-8))

def db(v):
    return 20 * np.log10(max(v, 1e-8))

def unit_f0(x):
    f0, vf, _ = librosa.pyin(x, fmin=70, fmax=400, sr=FS, frame_length=1024, hop_length=160)
    return np.where(vf & np.isfinite(f0), f0, np.nan)

def nanmed(a):
    a = np.asarray(a, dtype=float)
    return float(np.nanmedian(a)) if np.isfinite(a).any() else np.nan

n_edge = int(FS * CFG["join_ms"] / 1000)
t0 = time.perf_counter()
for w, lst in INV.items():
    for u in lst:
        x = u["x"]
        u["dur"] = len(x) / FS
        u["f0"] = unit_f0(x)
        u["f0_med"] = nanmed(u["f0"])
        u["f0_L"], u["f0_R"] = nanmed(u["f0"][:5]), nanmed(u["f0"][-5:])
        u["L"], u["R"] = edge_spec(x[:n_edge]), edge_spec(x[-n_edge:])
        u["rmsL"], u["rmsR"] = db(rms(x[:n_edge])), db(rms(x[-n_edge:]))
print(f"Trích đặc trưng {sum(len(v) for v in INV.values())} unit: {time.perf_counter()-t0:.1f}s")

all_units = [u for l in INV.values() for u in l]
vf = pd.DataFrame([dict(word=u["word"], take=u["take"], dur_ms=1000 * u["dur"], f0_med=u["f0_med"],
                        f0_L=u["f0_L"], f0_R=u["f0_R"]) for u in all_units])
vf.round(1).head(8)

## B. Baseline ghép nối (concatenation)

Ghép thẳng **một candidate cố định** (take 01) cho mỗi token, không cross-fade → `raw_concat.wav`. Đánh dấu vị trí biên nối để quan sát click, chênh energy, nhảy pitch và đổi đường bao phổ.

In [ ]:
def concat_with_bounds(xs, xfade_ms):
    # ghép + cross-fade tuyến tính (mục 3.6). Trả về waveform và vị trí tâm vùng nối (mẫu)
    n = int(FS * xfade_ms / 1000)
    y, bounds = xs[0].copy(), []
    for x in xs[1:]:
        m = min(n, len(y), len(x))
        if m == 0:
            bounds.append(len(y)); y = np.concatenate([y, x])
        else:
            a = np.linspace(0.0, 1.0, m, endpoint=True)
            mid = (1 - a) * y[-m:] + a * x[:m]
            bounds.append(len(y) - m + m // 2)
            y = np.concatenate([y[:-m], mid, x[m:]])
    return y.astype(np.float32), bounds

def show_spec(ax, y, title="", vlines=None, labels=None, fmax=8000):
    S = librosa.amplitude_to_db(np.abs(librosa.stft(y, n_fft=512, hop_length=160, win_length=400)), ref=np.max)
    ax.imshow(S, origin="lower", aspect="auto", cmap="magma", vmin=-80,
              extent=[0, len(y) / FS, 0, FS / 2000])
    ax.set_ylim(0, fmax / 1000); ax.set_ylabel("kHz"); ax.set_title(title)
    for v in (vlines or []):
        ax.axvline(v / FS, color="cyan", lw=0.8, ls="--")

tokens1 = normalize_queue_text(SENTENCES[0])
seq_fixed = [INV[w][0] for w in tokens1]
y_raw, b_raw = concat_with_bounds([u["x"] for u in seq_fixed], 0)
sf.write(OUT / "raw_concat.wav", y_raw, FS, subtype="PCM_16")
print("raw_concat.wav:", round(len(y_raw) / FS, 2), "s; số biên nối:", len(b_raw))

fig, ax = plt.subplots(2, 1, figsize=(13, 5.5), sharex=True)
t = np.arange(len(y_raw)) / FS
ax[0].plot(t, y_raw, lw=0.5, color="k")
edges = [0] + b_raw + [len(y_raw)]
for i, w in enumerate(tokens1):
    ax[0].text((edges[i] + edges[i + 1]) / 2 / FS, 0.9 * np.max(np.abs(y_raw)), w, ha="center", fontsize=9, color="tab:red")
for b in b_raw:
    ax[0].axvline(b / FS, color="tab:red", ls="--", lw=0.8)
ax[0].set_title("raw_concat.wav – waveform (đường đứt đỏ = biên nối)")
show_spec(ax[1], y_raw, "Spectrogram", vlines=b_raw); ax[1].set_xlabel("giây")
plt.tight_layout(); plt.savefig(FIG / "B_raw_concat.png", dpi=130); plt.show()

**Ghi nhận nghe raw_concat.wav (điền sau khi nghe):** click tại biên …; chênh energy giữa các từ …; nhảy pitch …; đổi đường bao phổ …; nhịp (không có khoảng lặng/ngữ điệu câu) …

## C. Unit selection và smoothing

Mục 3.5 của đề: chọn chuỗi unit $\theta^*=\arg\min \big[\sum_j C_{target}(\theta_j,t_j)+\lambda\sum_j C_{join}(\theta_j,\theta_{j+1})\big]$ bằng **quy hoạch động (Viterbi trên lưới candidate)**.

- **Target cost** (đo mức phù hợp với mục tiêu prosody): $C_{target}=w_d\,\frac{|\ln(d_\theta/d_t)|}{0.15}+w_f\,\frac{|12\log_2(F0_\theta/F0_t)|}{2\,\text{st}}$. Mục tiêu $d_t$, $F0_t$ lấy từ trung vị của từ đó (RMS đã chuẩn hoá nên không đưa vào target cost mà được xét trong join cost qua $|\Delta RMS|$); **từ cuối câu** được kéo dài ×1.15 và hạ giọng ×0.90 (phrase-final).
- **Join cost** (đo độ "êm" tại biên): $D_{spec}$ = sai khác bình phương trung bình của log-phổ 25 ms hai phía biên (công thức mục 5.4) + $|\Delta RMS|$ (dB) + $|\Delta F0|$ (semitone, nếu cả hai hữu thanh); mỗi thành phần chia cho độ lệch chuẩn của nó trên mọi cặp unit (chuẩn hoá $\sigma_k^2$ như mục 3.5).

In [ ]:
def join_parts(a, b):
    d_spec = float(np.mean((a["R"] - b["L"]) ** 2))                 # D_spec (dB^2)
    d_rms = abs(a["rmsR"] - b["rmsL"])                              # dB
    fa, fb = a["f0_R"], b["f0_L"]
    d_f0 = abs(12 * np.log2(fa / fb)) if (np.isfinite(fa) and np.isfinite(fb)) else 0.0   # semitone
    return d_spec, d_rms, d_f0

_pairs = np.array([join_parts(a, b) for a in all_units for b in all_units if a["word"] != b["word"]])
SIG = _pairs.std(axis=0) + 1e-9
print("σ chuẩn hoá (spec, rms, f0):", np.round(SIG, 2))

def join_cost(a, b):
    p = np.array(join_parts(a, b)) / SIG
    return float(CFG["w_spec"] * p[0] + CFG["w_rms"] * p[1] + CFG["w_f0"] * p[2])

WORD_STAT = {w: dict(dur=float(np.median([u["dur"] for u in l])), f0=nanmed([u["f0_med"] for u in l]))
             for w, l in INV.items()}

def make_targets(tokens):
    tg = []
    for j, w in enumerate(tokens):
        t = dict(dur=WORD_STAT[w]["dur"], f0=WORD_STAT[w]["f0"])
        if j == len(tokens) - 1:
            t["dur"] *= CFG["last_word_dur"]; t["f0"] *= CFG["last_word_f0"]
        tg.append(t)
    return tg

def target_cost(u, t):
    c = CFG["w_dur"] * abs(math.log(u["dur"] / t["dur"])) / CFG["dur_tol"]
    if np.isfinite(u["f0_med"]) and np.isfinite(t["f0"]):
        c += CFG["w_f0_target"] * abs(12 * math.log2(u["f0_med"] / t["f0"])) / CFG["f0_tol_st"]
    return c

def select_units(tokens, lam=None, w_target=1.0, n_cand=None):
    # Viterbi trên lưới candidate; trả về list unit đã chọn
    lam = CFG["lam"] if lam is None else lam
    tg = make_targets(tokens)
    cands = [(INV[w][:n_cand] if n_cand else INV[w]) for w in tokens]
    J = len(tokens)
    tc = [[w_target * target_cost(u, tg[j]) for u in cands[j]] for j in range(J)]
    best, back = [np.array(tc[0])], [None]
    for j in range(1, J):
        cur, bp = np.zeros(len(cands[j])), np.zeros(len(cands[j]), dtype=int)
        for k, u in enumerate(cands[j]):
            tot = [best[j - 1][kp] + lam * join_cost(cands[j - 1][kp], u) for kp in range(len(cands[j - 1]))]
            bp[k] = int(np.argmin(tot)); cur[k] = tc[j][k] + tot[bp[k]]
        best.append(cur); back.append(bp)
    k = int(np.argmin(best[-1])); path = [k]
    for j in range(J - 1, 0, -1):
        k = int(back[j][k]); path.append(k)
    path = path[::-1]
    return [cands[j][path[j]] for j in range(J)]

def eval_seq(seq, tokens, lam=None):
    lam = CFG["lam"] if lam is None else lam
    tg = make_targets(tokens)
    tcs = [target_cost(u, tg[j]) for j, u in enumerate(seq)]
    jcs = [join_cost(seq[j], seq[j + 1]) for j in range(len(seq) - 1)]
    return dict(target=sum(tcs), join=sum(jcs), total=sum(tcs) + lam * sum(jcs)), tcs, jcs

TOK = [normalize_queue_text(s) for s in SENTENCES]
STRATS = {
    "cố định take 01":       lambda tk: [INV[w][0] for w in tk],
    "chỉ target (λ=0)":      lambda tk: select_units(tk, lam=0.0),
    "chỉ join (w_target=0)": lambda tk: select_units(tk, w_target=0.0),
    "DP target + join":      lambda tk: select_units(tk),
}
rows, SEL = [], {}
for si, tk in enumerate(TOK):
    for name, fn in STRATS.items():
        seq = fn(tk); SEL[(si, name)] = seq
        ev, _, _ = eval_seq(seq, tk)
        rows.append(dict(câu=si + 1, cách_chọn=name, takes="-".join(str(u["take"]) for u in seq),
                         target=round(ev["target"], 2), join=round(ev["join"], 2), total=round(ev["total"], 2)))
cost_tbl = pd.DataFrame(rows)
cost_tbl.to_csv(RES / "C_unit_selection_costs.csv", index=False)
cost_tbl

In [ ]:
# Join cost từng biên: ít nhất 2 cách chọn candidate (câu 1)
tk = TOK[0]
cmp_rows = []
for name in ["cố định take 01", "DP target + join"]:
    _, _, jcs = eval_seq(SEL[(0, name)], tk)
    for j, c in enumerate(jcs):
        cmp_rows.append(dict(cách_chọn=name, biên=f"{tk[j]}|{tk[j+1]}", join_cost=round(c, 2)))
jt = pd.DataFrame(cmp_rows).pivot(index="biên", columns="cách_chọn", values="join_cost")
jt = jt.loc[[f"{tk[j]}|{tk[j+1]}" for j in range(len(tk) - 1)]]
jt["giảm"] = (jt["cố định take 01"] - jt["DP target + join"]).round(2)
print(jt.to_string())
jt.to_csv(RES / "C_join_cost_per_boundary_s1.csv")

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
x = np.arange(len(jt)); wd = 0.38
ax[0].bar(x - wd / 2, jt["cố định take 01"], wd, label="cố định take 01", color="#c44e52")
ax[0].bar(x + wd / 2, jt["DP target + join"], wd, label="DP target + join", color="#4c72b0")
ax[0].set_xticks(x); ax[0].set_xticklabels(jt.index, rotation=45, ha="right"); ax[0].set_ylabel("join cost")
ax[0].set_title("Join cost từng biên (câu 1)"); ax[0].legend()
p = cost_tbl[cost_tbl["câu"] == 1].set_index("cách_chọn")
ax[1].bar(p.index, p["target"], label="target", color="#55a868")
ax[1].bar(p.index, p["join"], bottom=p["target"], label="λ·join", color="#dd8452")
ax[1].set_title("Tổng chi phí theo cách chọn (câu 1)"); ax[1].legend(); plt.setp(ax[1].get_xticklabels(), rotation=20, ha="right")
plt.tight_layout(); plt.savefig(FIG / "C_unit_selection_costs.png", dpi=130); plt.show()

### Cross-fade và đo độ liên tục tại biên

Hai số đo khách quan tại mỗi biên nối:
- **click score** = $\max|\Delta y|$ trong ±3 ms quanh biên ÷ RMS của $\Delta y$ toàn câu (≫ 1 nghĩa là nhảy đột ngột, nghe thành click);
- **spectral jump** = khoảng cách log-phổ (dB, RMS) giữa hai cửa sổ 20 ms liền kề ngay trước/sau biên.

Lưu ý khi đọc số liệu: unit đã được trim kèm lề 10 ms và cắt tại zero-crossing nên biên thường rơi vào đoạn rất nhỏ, vì vậy **click score thường nhỏ (≈ 1 hoặc thấp hơn) ngay cả khi chưa cross-fade**; chỉ số phân biệt tốt hơn là spectral jump. Spectral jump cũng có xu hướng giảm đơn điệu theo độ dài cross-fade một phần vì cửa sổ đo nằm trong vùng trộn, nên phải đọc cùng `overlap_ms` (phần bị chồng, làm mờ phụ âm) và nghe thử.

Thí nghiệm **TN1**: cross-fade 0, 5, 15, 30 ms trên cả 3 câu (unit đã chọn bằng DP).

In [ ]:
def click_scores(y, bounds):
    d = np.diff(y); base = np.sqrt(np.mean(d ** 2)) + 1e-9; w = int(0.003 * FS)
    return [float(np.max(np.abs(d[max(b - w, 0):b + w])) / base) for b in bounds]

def spec_jump(y, bounds, ms=20):
    n = int(FS * ms / 1000); out = []
    for b in bounds:
        if b - n >= 0 and b + n <= len(y):
            out.append(float(np.sqrt(np.mean((edge_spec(y[b - n:b]) - edge_spec(y[b:b + n])) ** 2))))
        else:
            out.append(np.nan)
    return out

rows = []
for si, tk in enumerate(TOK):
    seq = SEL[(si, "DP target + join")]
    for xf in CFG["xfade_grid"]:
        y, b = concat_with_bounds([u["x"] for u in seq], xf)
        cs, sj = click_scores(y, b), spec_jump(y, b)
        rows.append(dict(câu=si + 1, xfade_ms=xf, click_mean=np.mean(cs), click_max=np.max(cs),
                         spec_jump_dB=np.nanmean(sj), overlap_ms=xf * (len(tk) - 1), dur_s=len(y) / FS))
        if si == 0:
            sf.write(OUT / "xfade" / f"s1_xfade{xf}ms.wav", y, FS, subtype="PCM_16")
tn1 = pd.DataFrame(rows)
tn1_mean = tn1.groupby("xfade_ms")[["click_mean", "click_max", "spec_jump_dB", "overlap_ms", "dur_s"]].mean().round(3)
tn1_mean.to_csv(RES / "TN1_crossfade.csv")
print("TN1 – trung bình trên 3 câu:")
print(tn1_mean.to_string())

In [ ]:
# best_concat.wav = câu 1, unit chọn bằng DP, cross-fade mặc định
seq_best = SEL[(0, "DP target + join")]
y_best, b_best = concat_with_bounds([u["x"] for u in seq_best], CFG["xfade_ms"])
sf.write(OUT / "best_concat.wav", y_best, FS, subtype="PCM_16")
print("best_concat.wav:", round(len(y_best) / FS, 2), "s; takes:", [u["take"] for u in seq_best])

# trước/sau cross-fade tại 2 biên có spectral jump lớn nhất khi chưa cross-fade (cùng chuỗi unit DP, xfade 0 vs mặc định)
y0, b0 = concat_with_bounds([u["x"] for u in seq_best], 0)
sj0 = np.nan_to_num(spec_jump(y0, b0), nan=-1.0)
pick = list(np.argsort(sj0)[::-1][:2])
y1, b1 = concat_with_bounds([u["x"] for u in seq_best], CFG["xfade_ms"])
half = int(0.08 * FS)
fig, ax = plt.subplots(4, 2, figsize=(13, 9))
for c, bi in enumerate(pick):
    for r, (yy, bb, tag) in enumerate([(y0, b0, "xfade 0 ms"), (y1, b1, f"xfade {CFG['xfade_ms']} ms")]):
        b = bb[bi]; lo, hi = max(b - half, 0), min(b + half, len(yy)); seg = yy[lo:hi]
        tt = (np.arange(lo, hi) - b) * 1000 / FS
        ax[2 * r, c].plot(tt, seg, lw=0.8, color="k"); ax[2 * r, c].axvline(0, color="r", ls="--", lw=0.8)
        ax[2 * r, c].set_title(f"{tag} – biên {TOK[0][bi]}|{TOK[0][bi+1]}")
        S = librosa.amplitude_to_db(np.abs(librosa.stft(seg, n_fft=256, hop_length=16, win_length=160)), ref=np.max)
        ax[2 * r + 1, c].imshow(S, origin="lower", aspect="auto", cmap="magma", vmin=-80,
                                extent=[tt[0], tt[-1], 0, 8]); ax[2 * r + 1, c].axvline(0, color="cyan", ls="--", lw=0.8)
        ax[2 * r + 1, c].set_ylabel("kHz")
    ax[3, c].set_xlabel("ms quanh biên")
plt.tight_layout(); plt.savefig(FIG / "C_boundary_before_after.png", dpi=130); plt.show()
print("spectral jump (dB) tại 2 biên đã chọn – xfade 0:", np.round([sj0[i] for i in pick], 1),
      "| xfade %d:" % CFG["xfade_ms"], np.round([spec_jump(y1, b1)[i] for i in pick], 1))

**Chọn cross-fade (điền sau khi nghe mù 0/5/15/30 ms, file ở `audio_output/xfade/`):** giá trị hợp lý nhất là … ms vì click score …, spectral jump giảm từ … xuống …, và nghe thấy phụ âm đầu từ … (bị mờ/không mờ). Cross-fade 30 ms làm tổng thời lượng ngắn đi … ms nhưng …

**TN2 – số candidate/unit** (1 so với 2–3 take): xem ô tiếp theo.

In [ ]:
rows = []
for n_c in [1, 2, 3]:
    for si, tk in enumerate(TOK):
        seq = select_units(tk, n_cand=n_c)
        ev, _, _ = eval_seq(seq, tk)
        rows.append(dict(n_candidate=n_c, câu=si + 1, target=ev["target"], join=ev["join"], total=ev["total"]))
tn2 = pd.DataFrame(rows).groupby("n_candidate")[["target", "join", "total"]].mean().round(2)
tn2.to_csv(RES / "TN2_candidates.csv")
print("TN2 – chi phí trung bình trên 3 câu theo số candidate/unit:")
print(tn2.to_string())

## D. Prosody và mô hình source–filter

**D1. Kiểm chứng $F_0 \approx F_s/N_p$** trên một đoạn gần tuần hoàn của unit hữu thanh nhất: đo chu kỳ pitch $N_p$ (mẫu) bằng tự tương quan 40 ms, so với $F_0$ của pyin.

In [ ]:
w_v, u_v = max(((w, u) for w, l in INV.items() for u in l), key=lambda t: np.isfinite(t[1]["f0"]).mean())
x_v, f0_v = u_v["x"], u_v["f0"]
vidx = np.where(np.isfinite(f0_v))[0]
rows = []
for pos in np.percentile(vidx, [30, 50, 70]).astype(int):
    c = pos * 160; seg = x_v[max(c - 320, 0):c + 320]; seg = seg - seg.mean()
    r = np.correlate(seg, seg, "full")[len(seg) - 1:]
    lo, hi = int(FS / 400), int(FS / 70)
    Np = lo + int(np.argmax(r[lo:hi]))
    rows.append(dict(đoạn=f"frame {pos}", Np_mau=Np, F0_tu_Np=round(FS / Np, 1), F0_pyin=round(float(f0_v[pos]), 1),
                     sai_lech_pct=round(100 * abs(FS / Np - f0_v[pos]) / f0_v[pos], 1)))
f0_check = pd.DataFrame(rows)
print(f"Unit dùng: '{w_v}' take {u_v['take']} ({100*np.isfinite(f0_v).mean():.0f}% frame hữu thanh)")
print(f0_check.to_string(index=False))
f0_check.to_csv(RES / "D_f0_check.csv", index=False)

fig, ax = plt.subplots(1, 2, figsize=(12, 3.2))
c = int(np.percentile(vidx, 50)) * 160; seg = x_v[max(c - 320, 0):c + 320]
ax[0].plot(np.arange(len(seg)) / FS * 1000, seg, color="k"); ax[0].set_title("Đoạn 40 ms gần tuần hoàn"); ax[0].set_xlabel("ms")
rr = np.correlate(seg - seg.mean(), seg - seg.mean(), "full")[len(seg) - 1:]
ax[1].plot(np.arange(len(rr)), rr); ax[1].axvline(rows[1]["Np_mau"], color="r", ls="--")
ax[1].set_title(f"Tự tương quan: đỉnh tại Np = {rows[1]['Np_mau']} mẫu → F0 = Fs/Np = {rows[1]['F0_tu_Np']} Hz"); ax[1].set_xlabel("độ trễ (mẫu)")
plt.tight_layout(); plt.savefig(FIG / "D_f0_period.png", dpi=130); plt.show()

**D2. Duration (TN3) và pitch (TN4).** Thay tốc độ bằng time-stretch (phase vocoder của librosa) với tốc độ 0.8×/1.0×/1.2×, và thay pitch ±2 semitone ($r=2^{\Delta s/12}$) bằng hai cách:
1. `librosa.effects.pitch_shift` (resample + time-stretch → **đổi cả đường bao phổ/formant**);
2. **TD-PSOLA tự cài** (cắt đoạn 2 chu kỳ quanh pitch mark, nhân cửa sổ Hann, đặt lại với khoảng cách chu kỳ mới → đổi $F_0$ nhưng giữ đường bao phổ tương đối).

Chỉ số: tỷ số $F_0$ đo được so với $r$ kỳ vọng, và **LSD đường bao** (khoảng cách đường bao phổ trung bình dài hạn, làm trơn bằng cepstral lifter, so với bản gốc; càng nhỏ càng ít méo formant).

In [ ]:
def psola_shift(y, semitones):
    # TD-PSOLA: đổi F0 theo hệ số beta = 2^(st/12), giữ nguyên thời lượng
    beta = 2 ** (semitones / 12)
    n = len(y)
    f0, vf, _ = librosa.pyin(y, fmin=70, fmax=400, sr=FS, frame_length=1024, hop_length=160)
    f0 = np.where(vf & np.isfinite(f0), f0, np.nan)
    default = int(0.008 * FS)
    sgn = 1.0 if np.max(y) >= -np.min(y) else -1.0
    marks, voiced, pos = [], [], 0
    while pos < n:
        fi = min(pos // 160, len(f0) - 1)
        v = bool(np.isfinite(f0[fi]))
        T0 = int(FS / f0[fi]) if v else default
        if v:
            if voiced and voiced[-1]:
                lo, hi = max(pos - T0 // 4, marks[-1] + 1), min(pos + T0 // 4 + 1, n)
            else:
                lo, hi = pos, min(pos + T0, n)
            if hi > lo:
                pos = lo + int(np.argmax(sgn * y[lo:hi]))
        if marks and pos <= marks[-1]:
            pos = marks[-1] + 1
        marks.append(pos); voiced.append(v); pos += T0
    ma, vm = np.array(marks), np.array(voiced)
    out = np.zeros(n)
    ts = float(ma[0])
    while ts < n:
        i = int(np.argmin(np.abs(ma - ts)))
        left = ma[i] - ma[i - 1] if i > 0 else (ma[i + 1] - ma[i] if len(ma) > 1 else default)
        right = ma[i + 1] - ma[i] if i + 1 < len(ma) else left
        a, b = ma[i] - left, ma[i] + right
        win = np.hanning(left + right + 2)[1:-1]
        lo_s, hi_s = max(a, 0), min(b, n)
        piece = y[lo_s:hi_s] * win[lo_s - a: lo_s - a + (hi_s - lo_s)]
        t_a = int(round(ts)) - left + (lo_s - a)
        d0, d1 = max(t_a, 0), min(t_a + len(piece), n)
        if d1 > d0:
            out[d0:d1] += piece[d0 - t_a: d1 - t_a]
        ts += (left + right) / 2 / (beta if vm[i] else 1.0)
    out *= rms(y) / (rms(out) + 1e-12)
    return np.clip(out, -0.99, 0.99).astype(np.float32)

def f0_median(y):
    f0, vf, _ = librosa.pyin(y, fmin=60, fmax=500, sr=FS, frame_length=1024, hop_length=160)
    return nanmed(np.where(vf, f0, np.nan))

def lt_envelope(y, q=30):
    S = np.abs(librosa.stft(y, n_fft=1024, hop_length=160, win_length=400)) ** 2
    e = S.sum(0); S = S[:, e > 0.05 * np.max(e)]
    L = 10 * np.log10(S.mean(axis=1) + 1e-12)
    cep = np.fft.irfft(L); cep[q:-q] = 0
    return np.fft.rfft(cep).real

def env_lsd(y1, y2):
    e1, e2 = lt_envelope(y1), lt_envelope(y2)
    f = np.fft.rfftfreq(1024, 1 / FS); m = (f >= 100) & (f <= 4000)
    e1, e2 = e1[m] - e1[m].mean(), e2[m] - e2[m].mean()
    return float(np.sqrt(np.mean((e1 - e2) ** 2)))

# --- TN3: duration / speed ---
rows = []
for sp in [0.8, 1.0, 1.2]:
    ys = y_best if sp == 1.0 else librosa.effects.time_stretch(y_best, rate=sp)
    sf.write(OUT / "prosody" / f"speed_{sp}x.wav", ys, FS, subtype="PCM_16")
    rows.append(dict(speed=f"{sp}x", dur_s=round(len(ys) / FS, 2), ti_so_dur=round(len(ys) / len(y_best), 3),
                     F0_trung_vi=round(f0_median(ys), 1)))
tn3 = pd.DataFrame(rows); tn3.to_csv(RES / "TN3_speed.csv", index=False)
print("TN3 – time-stretch (F0 phải gần như không đổi, duration ∝ 1/speed):"); print(tn3.to_string(index=False))

# --- TN4: pitch ±2 semitone ---
f_ref = f0_median(y_best)
rows = []
for st in [-2, 0, 2]:
    for name, fn in [("librosa", lambda y, s: librosa.effects.pitch_shift(y, sr=FS, n_steps=s)),
                     ("TD-PSOLA", psola_shift)]:
        ys = y_best if st == 0 else fn(y_best, st)
        if st == 0 and name == "TD-PSOLA":
            continue
        sf.write(OUT / "prosody" / f"pitch_{name.lower().replace('-', '')}_{st:+d}st.wav", ys, FS, subtype="PCM_16")
        rows.append(dict(phương_pháp=name if st else "gốc", semitone=st, r_ky_vong=round(2 ** (st / 12), 3),
                         r_do_duoc=round(f0_median(ys) / f_ref, 3), LSD_duong_bao_dB=round(env_lsd(y_best, ys), 2)))
tn4 = pd.DataFrame(rows); tn4.to_csv(RES / "TN4_pitch.csv", index=False)
print("TN4 – pitch shift (r đo được ≈ r kỳ vọng; LSD thấp = ít méo đường bao):"); print(tn4.to_string(index=False))

fig, ax = plt.subplots(1, 3, figsize=(15, 3.6), sharey=True)
for a, (nm, fn) in zip(ax, [("gốc", None), ("librosa +2 st", "pitch_librosa_+2st.wav"), ("TD-PSOLA +2 st", "pitch_tdpsola_+2st.wav")]):
    yy = y_best if fn is None else sf.read(str(OUT / "prosody" / fn))[0]
    show_spec(a, yy, nm, fmax=4000)
plt.tight_layout(); plt.savefig(FIG / "D_pitch_shift_spectrograms.png", dpi=130); plt.show()

**Nhận xét (điền sau khi nghe `audio_output/prosody/`):** tốc độ 0.8×/1.2× … (intelligibility, naturalness); pitch ±2 st bằng librosa nghe … (formant dịch theo, có thể "chipmunk"/buzzy), TD-PSOLA nghe … Số liệu LSD đường bao ở bảng TN4 cho thấy …

**Vì sao đổi duration/pitch không nên đổi đường bao phổ tùy ý:** theo mô hình source–filter $X(z)=E(z)H(z)$, $F_0$ chỉ nằm ở nguồn kích thích $E(z)$ (khoảng cách xung), còn formant/đường bao phổ nằm ở bộ lọc cộng hưởng $H(z)$ phụ thuộc hình dạng đường dẫn thanh (nguyên âm, người nói). Nếu đổi pitch bằng cách resample thì cả $E$ lẫn $H$ cùng bị co giãn → formant dịch → nguyên âm nghe như nguyên âm khác/giọng trẻ con. Time-stretch cần giữ nguyên các frame phổ, chỉ lặp/bỏ frame.

## E. Tổng hợp tham số dựa trên HMM (bài quan sát có hướng dẫn)

Backend `hmm_tts_backend.py` (đi kèm repo) thực hiện đúng quy trình mục 3.7:

1. **Huấn luyện** từ chính inventory: mỗi từ một HMM trái-sang-phải 5 state; mỗi state là Gaussian chéo của $o_t=[c_t,\Delta c_t,\Delta^2 c_t]$ ($c_t$: mel-cepstrum 24 chiều), cộng log-F0, xác suất hữu thanh và duration (segmental k-means + căn chỉnh Viterbi).
2. **Generation**: câu → chuỗi từ → chuỗi state, mỗi state giữ `round(dur_state × duration_scale)` frame; quỹ đạo tĩnh $c^*=\arg\max_c P(Wc\mid\lambda)$ (MLPG), tức nghiệm của $(W^\top\Sigma^{-1}W)c=W^\top\Sigma^{-1}\mu$.
3. **Vocoder** source–filter: excitation (xung theo F0 / nhiễu) × đường bao phổ từ $c^*$ → STFT/OLA.

Không tái tạo từ MFCC chuẩn mà dùng mel-cepstrum + vocoder đơn giản nên giọng sẽ nghe "buzzy/robotic" – đó là đặc trưng của tổng hợp tham số.

In [ ]:
HMM_DIR = DATA / "hmm_demo"; HMM_DIR.mkdir(parents=True, exist_ok=True)
t0 = time.perf_counter()
models_hmm = HMMB.train_hmm_models(UNITS, HMM_DIR / "hmm_models.pkl", verbose=False)
print(f"Huấn luyện {len(models_hmm)} HMM: {time.perf_counter()-t0:.1f}s")
ex_word = "hai" if "hai" in models_hmm else next(iter(models_hmm))
print(f"Tham số duration (frame/state) của '{ex_word}':", np.round(models_hmm[ex_word]["dur"], 1))
print(f"Xác suất hữu thanh theo state của '{ex_word}':", np.round(models_hmm[ex_word]["vprob"], 2))

y_hmm, sr_h, tracks = HMMB.synthesize_hmm(SENTENCES[0], models=models_hmm)
y_hmm_nd, _, tr_nd = HMMB.synthesize_hmm(SENTENCES[0], models=models_hmm, use_dynamic=False)
sf.write(OUT / "hmm_tts.wav", y_hmm, sr_h, subtype="PCM_16")
sf.write(OUT / "hmm_tts_no_delta.wav", y_hmm_nd, sr_h, subtype="PCM_16")
print("hmm_tts.wav:", round(len(y_hmm) / sr_h, 2), "s,", len(tracks["f0"]), "frame; tokens:", tracks["tokens"])

T = len(tracks["f0"]); tt = np.arange(T) / tracks["frame_rate"]
chg = np.where((np.diff(tracks["states"]) != 0) & ~tracks["silence"][1:] & ~tracks["silence"][:-1])[0]
jump_dyn = np.abs(tracks["c"][chg + 1, 1:6] - tracks["c"][chg, 1:6]).mean()
jump_sta = np.abs(tr_nd["c"][chg + 1, 1:6] - tr_nd["c"][chg, 1:6]).mean()
print(f"Bước nhảy trung bình của c1..c5 tại biên state: có Δ/Δ² (MLPG) = {jump_dyn:.3f} | chỉ mean tĩnh = {jump_sta:.3f}"
      f"  (giảm {100*(1-jump_dyn/jump_sta):.0f}%)")

fig, ax = plt.subplots(4, 1, figsize=(13, 10), sharex=True)
f0p = np.where(tracks["f0_hz"] > 0, tracks["f0_hz"], np.nan)
ax[0].plot(tt, f0p, "o-", ms=2, color="tab:blue"); ax[0].set_ylabel("F0 (Hz)"); ax[0].set_title("Quỹ đạo F0 do HMM sinh (chỉ frame hữu thanh)")
ax[1].plot(tt, tr_nd["c"][:, 1], color="tab:orange", lw=1, label="chỉ mean tĩnh (bậc thang)")
ax[1].plot(tt, tracks["c"][:, 1], color="tab:green", lw=1.6, label="MLPG với Δ, Δ²")
ax[1].set_ylabel("c1 (mel-cepstrum)"); ax[1].legend(loc="upper right"); ax[1].set_title("Một tham số phổ: bậc thang vs quỹ đạo trơn")
ax[2].plot(tt, tracks["c"][:, 2], color="tab:green", label="c2 (MLPG)"); ax[2].plot(tt, tracks["c"][:, 3], color="tab:purple", label="c3 (MLPG)")
ax[2].set_ylabel("c2, c3"); ax[2].legend(loc="upper right")
ax[3].step(tt, tracks["states"], where="post", color="k"); ax[3].set_ylabel("state toàn cục"); ax[3].set_xlabel("giây")
ax[3].set_title("Chuỗi state (5 state/từ, chỉ số = từ×5 + state)")
for a in ax:
    for ws in tracks["word_start"]:
        a.axvline(ws / tracks["frame_rate"], color="gray", ls=":", lw=0.8)
for i, ws in enumerate(tracks["word_start"]):
    ax[0].text(ws / tracks["frame_rate"] + 0.02, np.nanmax(f0p) * 1.0, tracks["tokens"][i], fontsize=9, color="tab:red")
plt.tight_layout(); plt.savefig(FIG / "E_hmm_tracks.png", dpi=130); plt.show()

In [ ]:
# thí nghiệm: scale duration / F0 trên backend HMM (resynthesis)
rows = []
for name, kw in [("gốc", {}), ("duration x0.8", dict(duration_scale=0.8)), ("duration x1.2", dict(duration_scale=1.2)),
                 ("F0 x0.9", dict(f0_scale=0.9)), ("F0 x1.1", dict(f0_scale=1.1))]:
    yy, srr, tr = HMMB.synthesize_hmm(SENTENCES[0], models=models_hmm, **kw)
    fn = "hmm_" + name.replace(" ", "_").replace("x", "x") + ".wav"
    sf.write(OUT / "prosody" / fn, yy, srr, subtype="PCM_16")
    rows.append(dict(cấu_hình=name, dur_s=round(len(yy) / srr, 2), số_frame=len(tr["f0"]),
                     F0_trung_vi_Hz=round(float(np.median(tr["f0_hz"][tr["f0_hz"] > 0])), 1)))
tn_hmm = pd.DataFrame(rows); tn_hmm.to_csv(RES / "E_hmm_param_scaling.csv", index=False)
print(tn_hmm.to_string(index=False))

**Giải thích (mục 4.5):**
- *Chọn waveform tự nhiên* (concatenation): lấy nguyên đoạn tiếng nói thật, nên đường bao phổ, nhiễu hơi thở, biến thiên vi mô của giọng đều tự nhiên; điểm yếu là biên nối và phải có đủ unit cho mọi ngữ cảnh.
- *Sinh quỹ đạo tham số từ phân bố HMM*: không phát lại mẫu nào; mỗi state chỉ lưu **trung bình/phương sai**, quỹ đạo là nghiệm kỳ vọng nên **được làm trơn (trung bình hóa)** → liên tục tại biên, nhỏ gọn, sửa được duration/F0 tùy ý, nhưng mất chi tiết vi mô (over-smoothing) và vocoder làm giọng buzzy.
- **Δ, Δ² làm trajectory trơn**: nếu chỉ tối đa hóa $P(c\mid\lambda)$ theo mean tĩnh thì $c^*=\mu_{state}$ là bậc thang. Khi bổ sung Δ, Δ², ta tối đa hóa $P(Wc\mid\lambda)$ trong đó $Wc$ ràng buộc các sai phân liên tiếp; vì Gaussian của Δ có trung bình ≈ 0 (đặc biệt ở state ổn định), nghiệm phải đánh đổi giữa "gần mean tĩnh" và "ít thay đổi giữa frame", nên bậc thang bị bo tròn. Số liệu ở trên: bước nhảy tại biên state giảm theo phần trăm đã in.

## F. TTS học sâu bằng mô hình pretrained (offline)

Wrapper `neural_tts_backend.py` dùng **Piper (VITS, ONNX)**. Pipeline: văn bản (đã chuẩn hóa thành chữ) → phonemize bằng eSpeak-ng (front-end) → mô hình VITS sinh biểu diễn âm học/latent và waveform qua decoder kiểu vocoder (HiFi-GAN-like) → waveform. Model phải được đóng gói **trước** giờ Lab (không tải ngẫu nhiên từ Internet) – xem README.

Nếu chưa có model trong `lab4_data/neural/`, ô dưới sẽ báo rõ và phần neural được đánh dấu "chưa chạy"; các phần còn lại vẫn hoàn chỉnh.

In [ ]:
NN_TEXT = [" ".join(t).capitalize() + "." for t in TOK]     # "Mời khách hàng số hai ba đến quầy năm."
NEURAL_OK, nn_out = False, []
try:
    import neural_tts_backend as NNB
    NNB.synthesize(NN_TEXT[0])                              # khởi động: nạp model (không tính vào latency)
    for txt in NN_TEXT:
        nn_out.append(NNB.synthesize(txt))
    NEURAL_OK = True
except Exception as e:
    print("[Neural TTS chưa chạy]", type(e).__name__, "-", str(e).splitlines()[0])

if NEURAL_OK:
    y_nn, sr_nn, ex0 = nn_out[0]
    sf.write(OUT / "neural_tts.wav", y_nn, sr_nn, subtype="PCM_16")
    lat = [o[2]["latency_s"] for o in nn_out]
    print(f"neural_tts.wav: {len(y_nn)/sr_nn:.2f} s @ {sr_nn} Hz | model {ex0['model']} ({ex0['model_size_mb']:.0f} MB), nạp model {ex0['load_time_s']:.2f}s")
    print("Latency 3 câu (s):", np.round(lat, 3), "| RTF:", np.round([o[2]["rtf"] for o in nn_out], 3))
    fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
    ax[0].imshow(ex0["mel"], origin="lower", aspect="auto", cmap="magma"); ax[0].set_title("Mel-spectrogram (tính từ waveform neural)")
    show_spec(ax[1], y_nn if sr_nn == FS else librosa.resample(y_nn, orig_sr=sr_nn, target_sr=FS), "Spectrogram neural_tts.wav")
    plt.tight_layout(); plt.savefig(FIG / "F_neural_mel.png", dpi=130); plt.show()

## G. Đánh giá và so sánh ba phương pháp

- **Khách quan** (tự động): latency (từ text → waveform, trên cùng máy), **continuity** (tỷ lệ phân vị 99 / trung vị của spectral flux giữa các frame – càng gần 1 càng đều, càng lớn càng có "nhảy"), bộ nhớ/dữ liệu.
- **Chủ quan** (cần người nghe): intelligibility (% từ nghe đúng), naturalness và continuity (thang 1–5). Notebook tạo sẵn file nghe **mù** `audio_output/blind/*.wav` (tên ngẫu nhiên, cùng loudness) và `ratings.csv` mẫu; điền kết quả nghe của 5–10 người rồi chạy lại ô cuối để cập nhật bảng.

In [ ]:
def flux_ratio(y):
    M = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=FS, n_fft=512, hop_length=160, n_mels=40, power=2.0), ref=np.max)
    act = M.max(axis=0) > -45
    f = np.sqrt(np.mean(np.diff(M, axis=1) ** 2, axis=0))
    a = act[1:] & act[:-1]
    return float(np.percentile(f[a], 99) / (np.median(f[a]) + 1e-9)) if a.sum() > 5 else np.nan

def to16k(y, sr):
    return y if sr == FS else librosa.resample(y, orig_sr=sr, target_sr=FS)

METHODS = ["Concatenation", "HMM/parametric"] + (["Neural TTS"] if NEURAL_OK else [])
OUTS = {m: [] for m in METHODS}
for si in range(len(SENTENCES)):
    t0 = time.perf_counter()
    seq = select_units(TOK[si]); yc, _ = concat_with_bounds([u["x"] for u in seq], CFG["xfade_ms"])
    OUTS["Concatenation"].append((yc, time.perf_counter() - t0))
    t0 = time.perf_counter()
    yh, _, _ = HMMB.synthesize_hmm(SENTENCES[si], models=models_hmm)
    OUTS["HMM/parametric"].append((yh, time.perf_counter() - t0))
    if NEURAL_OK:
        yn, srn, exn = nn_out[si]
        OUTS["Neural TTS"].append((to16k(yn, srn), exn["latency_s"]))

inv_mb = sum(Path(u["file"]).stat().st_size for u in all_units) / 1e6
size_mb = {"Concatenation": inv_mb,
           "HMM/parametric": (HMM_DIR / "hmm_models.pkl").stat().st_size / 1e6,
           "Neural TTS": (nn_out[0][2]["model_size_mb"] if NEURAL_OK else np.nan)}
obj = pd.DataFrame([dict(Phương_pháp=m,
                         Continuity_flux_p99_over_median=round(np.nanmean([flux_ratio(y) for y, _ in OUTS[m]]), 2),
                         Latency_s=round(np.mean([l for _, l in OUTS[m]]), 3),
                         Bộ_nhớ_dữ_liệu_MB=round(size_mb[m], 2)) for m in METHODS])
print("Chỉ số khách quan (trung bình 3 câu; latency concat không tính trích đặc trưng unit làm sẵn):")
print(obj.to_string(index=False))

# --- tạo file nghe mù + ratings.csv mẫu ---
def norm_active(y, target=0.08):
    a = np.abs(y) > 0.05 * np.max(np.abs(y))
    return np.clip(y * (target / rms(y[a])), -0.99, 0.99).astype(np.float32)

rng = np.random.default_rng(2025)
items = [(m, si) for m in METHODS for si in range(len(SENTENCES))]
codes = rng.permutation(len(items))
key_rows = []
for (m, si), c in zip(items, codes):
    code = f"S{c+1:02d}"
    sf.write(OUT / "blind" / f"{code}.wav", norm_active(OUTS[m][si][0]), FS, subtype="PCM_16")
    key_rows.append(dict(code=code, câu=si + 1, phương_pháp=m, văn_bản_chuẩn=" ".join(TOK[si])))
key = pd.DataFrame(key_rows).sort_values("code")
key.to_csv(RES / "blind_key.csv", index=False)           # KHÔNG đưa file này cho người nghe
if not Path("ratings.csv").exists():
    pd.DataFrame(columns=["listener", "code", "heard_text", "naturalness_1_5", "continuity_1_5"]).to_csv("ratings.csv", index=False)
print(f"\nĐã tạo {len(key)} file nghe mù trong audio_output/blind/ (khóa: results/blind_key.csv) và ratings.csv mẫu.")

In [ ]:
# --- đọc ratings.csv (nếu đã có người nghe) và hoàn thiện bảng so sánh ---
def wer(ref, hyp):
    r, h = ref.split(), hyp.split()
    d = np.zeros((len(r) + 1, len(h) + 1), dtype=int)
    d[:, 0], d[0, :] = np.arange(len(r) + 1), np.arange(len(h) + 1)
    for i in range(1, len(r) + 1):
        for j in range(1, len(h) + 1):
            d[i, j] = min(d[i - 1, j] + 1, d[i, j - 1] + 1, d[i - 1, j - 1] + (r[i - 1] != h[j - 1]))
    return d[-1, -1] / max(len(r), 1)

comp = obj.copy()
comp["Intelligibility_%"] = np.nan; comp["Naturalness_MOS"] = np.nan; comp["Continuity_MOS"] = np.nan
rat = pd.read_csv("ratings.csv") if Path("ratings.csv").exists() else pd.DataFrame()
if len(rat) and rat["naturalness_1_5"].notna().any():
    rat = rat.merge(key, on="code", how="left")
    rat["intel"] = [1 - min(1.0, wer(ref, " ".join(normalize_queue_text(str(h)))))
                    if isinstance(h, str) else np.nan for ref, h in zip(rat["văn_bản_chuẩn"], rat["heard_text"])]
    g = rat.groupby("phương_pháp").agg(I=("intel", "mean"), N=("naturalness_1_5", "mean"), C=("continuity_1_5", "mean"),
                                       n=("listener", "count"))
    for i, r_ in comp.iterrows():
        if r_["Phương_pháp"] in g.index:
            comp.loc[i, "Intelligibility_%"] = round(100 * g.loc[r_["Phương_pháp"], "I"], 1)
            comp.loc[i, "Naturalness_MOS"] = round(g.loc[r_["Phương_pháp"], "N"], 2)
            comp.loc[i, "Continuity_MOS"] = round(g.loc[r_["Phương_pháp"], "C"], 2)
    print(f"Đã đọc {len(rat)} lượt nghe từ {rat['listener'].nunique()} người.")
else:
    print("ratings.csv chưa có dữ liệu nghe -> các cột chủ quan để NaN (điền ratings.csv rồi chạy lại ô này).")
comp.to_csv(RES / "G_comparison.csv", index=False)
comp

In [ ]:
fig, ax = plt.subplots(len(METHODS), 1, figsize=(12, 3.0 * len(METHODS)), sharex=True)
ax = np.atleast_1d(ax)
for a, m in zip(ax, METHODS):
    show_spec(a, OUTS[m][0][0], f"{m} – câu 1", fmax=8000)
ax[-1].set_xlabel("giây")
plt.tight_layout(); plt.savefig(FIG / "G_method_comparison.png", dpi=130); plt.show()

**Nhận xét so sánh (điền sau khi nghe và xem số liệu):**

| Phương pháp | Intelligibility | Naturalness | Continuity | Latency | Bộ nhớ/dữ liệu |
|---|---|---|---|---|---|
| Concatenation | … | … (giọng thật nhưng nhịp "đọc từng từ", ngắt biên) | … (phụ thuộc join) | thấp (chọn + nối) | cần inventory đủ rộng |
| HMM/parametric | … | … (buzzy, over-smoothing) | … (quỹ đạo tham số trơn nhờ Δ/Δ², nhưng chỉ số flux ở mức waveform còn chịu ảnh hưởng của nhiễu/xung vocoder – đối chiếu số đo) | thấp | nhỏ (chỉ tham số state) |
| Neural TTS | … | … | … | cao nhất (suy luận mạng) | model hàng chục MB |

## Thí nghiệm bắt buộc (tóm tắt) và câu hỏi báo cáo

| TN | Biến | Kết quả nằm ở |
|---|---|---|
| 1 | cross-fade 0/5/15/30 ms | `results/TN1_crossfade.csv`, `figures/C_boundary_before_after.png`, `audio_output/xfade/` |
| 2 | 1 vs 2–3 candidate/unit | `results/TN2_candidates.csv`, `results/C_unit_selection_costs.csv` |
| 3 | duration 0.8/1.0/1.2× | `results/TN3_speed.csv`, `audio_output/prosody/speed_*` |
| 4 | pitch −2/0/+2 semitone | `results/TN4_pitch.csv`, `figures/D_pitch_shift_spectrograms.png` |
| 5 | concat / HMM / neural | `results/G_comparison.csv`, `figures/G_method_comparison.png` |

### Trả lời câu hỏi báo cáo (mục 7)

**1. Vì sao câu ghép từ đoạn "tự nhiên" vẫn nghe kém tự nhiên?** Mỗi từ được thu **riêng lẻ** nên mang ngữ điệu/trọng âm/duration của từ đứng một mình, không có hòa âm (coarticulation) và ngữ điệu liên câu (F0 đầu/cuối câu, kéo dài cuối cụm). Tại biên còn có nhảy mức, nhảy pitch, đổi đường bao phổ và không có chuyển tiếp hình thái thanh quản/âm vị. Do đó từng đoạn tự nhiên nhưng *chuỗi* không tự nhiên.

**2. Target cost và join cost; chỉ tối ưu join cost thì sao?** Target cost đo unit có **phù hợp mục tiêu** (duration, F0 cần cho vị trí/ngữ cảnh) không; join cost đo **độ êm tại biên** giữa hai unit liên tiếp. Chỉ tối ưu join cost sẽ chọn các unit nối rất êm nhưng sai prosody (quá ngắn/dài, sai cao độ, từ cuối câu không hạ giọng) – bảng "chỉ join" trong ô C cho thấy target cost tăng so với DP.

**3. Vì sao cross-fade quá dài làm giảm độ rõ phụ âm?** Phụ âm (bật hơi, xát, tắc) rất ngắn (vài chục ms) và mang thông tin ở **chuyển tiếp** đầu/cuối từ. Cross-fade dài cộng chồng cả đoạn này với từ kế bên: phụ âm bị suy giảm biên độ, trộn lẫn hai phổ nên "mờ", và tổng thời lượng bị rút ngắn.

**4. Vì sao thay F0 không đồng nghĩa thay formant (source–filter)?** $X(z)=E(z)H(z)$: $F_0$ là tần số lặp của nguồn $E$; formant là các cực của vocal-tract filter $H$. Đổi khoảng cách xung giữ nguyên $H$ thì F0 đổi mà formant giữ nguyên (đổi cao độ nhưng vẫn là cùng một nguyên âm/người nói). Chỉ khi ta co giãn phổ (resample) thì formant mới dịch theo (xem LSD đường bao ở TN4).

**5. PSOLA đổi pitch thế nào? Vì sao voiced fricative bị buzzy khi kéo dài quá mức?** TD-PSOLA cắt các đoạn quanh pitch mark (≈2 chu kỳ, nhân cửa sổ) rồi đặt lại với khoảng cách chu kỳ mới (khoảng cách nhỏ → F0 cao) và lặp/bỏ đoạn để giữ duration, nên đường bao phổ trong từng đoạn gần như giữ nguyên. Voiced fricative (z, v…) gồm thành phần nhiễu + thành phần hữu thanh; khi kéo dài nhiều, cùng một vài chu kỳ bị **lặp lại** nhiều lần → nhiễu cũng được lặp tuần hoàn → nghe buzzy/tonal.

**6. Vì sao thêm delta feature làm trajectory HMM trơn hơn?** Xem phần E: MLPG tối đa hóa $P(Wc\mid\lambda)$ có ràng buộc Δ, Δ²; thay vì bậc thang theo mean của state, nghiệm là đường cong đánh đổi giữa mean tĩnh và độ biến thiên cho phép. Số liệu bước nhảy tại biên state (ô E) cho thấy giảm rõ rệt.

**7. Vì sao concatenation cần inventory lớn để đạt naturalness cao?** Càng nhiều bản ghi (nhiều ngữ cảnh, prosody, duration) thì càng có khả năng tìm unit **vừa khớp target vừa nối êm** (cả target và join cost đều thấp). Inventory nhỏ (2–3 take/từ như Lab này) hầu như không có lựa chọn → TN2 chỉ cải thiện nhẹ.

**8. Hai khác biệt cơ bản giữa concatenative và neural TTS.** (i) Concatenative *phát lại mẫu thật* được chọn/ghép theo luật chi phí, còn neural *sinh* waveform từ biểu diễn học được (không lưu mẫu); (ii) concatenative cần inventory lớn và thuật toán chọn thủ công, còn neural học prosody, coarticulation và đường bao trực tiếp từ dữ liệu huấn luyện, tổng quát hóa sang câu chưa thấy và liên tục tự nhiên, đổi lại cần nhiều dữ liệu/tính toán và khó can thiệp từng tham số.

**9. Vì sao không so sánh neural TTS bằng model tùy ý tải từ Internet trong giờ Lab?** Kết quả không tái lập (khác model/phiên bản/thiết bị, như bảng lỗi mục 10), tốn băng thông/thời gian, có rủi ro an toàn (file lạ) và không công bằng (model huấn luyện trên dữ liệu rất khác miền hẹp). Vì vậy đóng gói sẵn model + version trước giờ Lab.

**10. MOS cao nhưng intelligibility thấp thì dùng vào đâu?** Giọng nghe hay/dễ chịu nhưng khó nghe ra nội dung → phù hợp ứng dụng **giải trí, đọc sách nói thư giãn, nhân vật ảo, quảng cáo thương hiệu** (cần cảm xúc/âm sắc hơn là chính xác từng từ), **không** phù hợp thông báo an toàn, xếp hàng, hướng dẫn y tế/giao thông – những nơi nghe đúng nội dung là ưu tiên.

In [ ]:
print("=" * 60); print("TÓM TẮT KẾT QUẢ LAB 4"); print("=" * 60)
print(f"Inventory: {len(INV)} từ, {len(all_units)} unit, {inv_mb:.2f} MB")
print("\n[C] chi phí unit selection (câu 1):"); print(cost_tbl[cost_tbl['câu'] == 1].to_string(index=False))
print("\n[TN1] cross-fade (trung bình 3 câu):"); print(tn1_mean.to_string())
print("\n[TN2] số candidate/unit:"); print(tn2.to_string())
print("\n[TN3] speed:"); print(tn3.to_string(index=False))
print("\n[TN4] pitch:"); print(tn4.to_string(index=False))
print("\n[E] HMM scaling:"); print(tn_hmm.to_string(index=False))
print("\n[G] so sánh:"); print(comp.to_string(index=False))
print("\nFile âm thanh:", sorted(p.name for p in OUT.glob("*.wav")))